In [128]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [129]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [130]:
conversion_rates          = pd.read_csv('/content/drive/MyDrive/CS412/HW2/CSVs/conversionRates.csv')
free_form_responses       = pd.read_csv('/content/drive/MyDrive/CS412/HW2/CSVs/freeformResponses.csv')
multiple_choice_responses = pd.read_csv('/content/drive/MyDrive/CS412/HW2/CSVs/multipleChoiceResponses.csv', encoding="ISO-8859-1")
schema                    = pd.read_csv('/content/drive/MyDrive/CS412/HW2/CSVs/schema.csv')



<ipython-input-130-c1dc1fd39191>:2: DtypeWarning: Columns (5,17,21,38,50) have mixed types. Specify dtype option on import or set low_memory=False.
  free_form_responses       = pd.read_csv('/content/drive/MyDrive/CS412/HW2/CSVs/freeformResponses.csv')
<ipython-input-130-c1dc1fd39191>:3: DtypeWarning: Columns (31,83,86,87,98,99,109,116,123,124,127,129,130,164) have mixed types. Specify dtype option on import or set low_memory=False.
  multiple_choice_responses = pd.read_csv('/content/drive/MyDrive/CS412/HW2/CSVs/multipleChoiceResponses.csv', encoding="ISO-8859-1")


In [131]:
mcr = multiple_choice_responses.copy()

mcr["CompensationAmount"] = mcr[mcr["CompensationAmount"].notna()]["CompensationAmount"].str.replace(',', '')
mcr["CompensationAmount"] = mcr["CompensationAmount"].replace("-", np.nan)
mcr["CompensationAmount"] = mcr[mcr["CompensationAmount"].notna()]["CompensationAmount"].astype(float)

extracted_columns = conversion_rates.loc[:, ['originCountry', 'exchangeRate']]
extracted_columns = extracted_columns.rename(columns={'originCountry': "CompensationCurrency"})

mcr = mcr.join(extracted_columns.set_index("CompensationCurrency"), on="CompensationCurrency")

mcr["ConvertedSalary"] = mcr["CompensationAmount"] * mcr["exchangeRate"]

In [143]:

cpp_users = len(mcr[mcr["WorkToolsSelect"].str.contains("C\+\+", na=False, case=False)])

pr_and_cpp = len(mcr[(mcr['CurrentJobTitleSelect'] == "Programmer") & (mcr["WorkToolsSelect"].str.contains("C\+\+", na=False, case=False))])

pr_given_cpp = pr_and_cpp / cpp_users

print("Probability of c++ user being a programmer:", pr_given_cpp)

Probability of c++ user being a programmer: 0.02225130890052356


In [133]:

majors = mcr[mcr['MajorSelect'].str.contains('mathematics|computer science|statistics', na=False, case=False)]

total_majors  = len(majors)

data_scientists_with_majors = len(majors[majors['CurrentJobTitleSelect'] == 'Data Scientist'])

data_scientists_prob  = data_scientists_with_majors  / total_majors

print("Probability of math, comp or statistics major being a data scientist:", data_scientists_prob)

Probability of math, comp or statistics major being a data scientist: 0.1597400634728729


In [134]:
salary_gt_40k = mcr[mcr["ConvertedSalary"] > 40000]

total_salary_gt_40k = len(salary_gt_40k)

tech_salary_gt_40k = len(salary_gt_40k[salary_gt_40k["EmployerIndustry"] == "Technology"])

tech_salary_gt_40k_prob = tech_salary_gt_40k / total_salary_gt_40k

print("Probability that a respondent works in the Technology industry given that they earn more than 40,000 USD:", tech_salary_gt_40k_prob)

Probability that a respondent works in the Technology industry given that they earn more than 40,000 USD: 0.1935483870967742


In [135]:
education_filter = ["Bachelor's degree", "Master's degree", 'Doctoral degree', 'Professional degree']

gt_30_educated = len(mcr[(mcr["Age"] > 30.0) & (mcr["FormalEducation"].isin(education_filter))])

total = len(mcr[mcr["FormalEducation"].notna()])

gt_30_educated_prob = gt_30_educated / total

print("Joint probability of a respondent being over 30 years old and having a at least a Bachelors degree:", gt_30_educated_prob)

Joint probability of a respondent being over 30 years old and having a at least a Bachelors degree: 0.4307026307026307


In [136]:
data_scientist_with_major= len(mcr[((mcr['MajorSelect'] == "Computer Science") | (mcr['MajorSelect'] == "Mathematics or statistics"))
                                  & (mcr['CurrentJobTitleSelect'] == 'Data Scientist')])

respondents =len(mcr[mcr['MajorSelect'].notna()])

data_scientist_with_major_prob = data_scientist_with_major / respondents

print("Probability that a respondent is a Data Scientist who majored in Computer Science, Mathematics or statistics:", data_scientist_with_major_prob)

Probability that a respondent is a Data Scientist who majored in Computer Science, Mathematics or statistics: 0.07958738046833823


In [137]:
#& ((mcr['WorkMethodsFrequencyCross-Validation'] == "Often") | (mcr['WorkMethodsFrequencyCross-Validation'] == "Most of the time"))

france_lt_100k_cross = len(mcr[(mcr["Country"] == "France")
                        & (mcr["ConvertedSalary"] < 100000)
                        & ((mcr['WorkMethodsFrequencyCross-Validation'] == "Often") | (mcr['WorkMethodsFrequencyCross-Validation'] == "Most of the time"))])

cross_respondents = len(mcr[mcr['WorkMethodsFrequencyCross-Validation'].notna()])

france_lt_100k_cross_prob = france_lt_100k_cross / cross_respondents

print("Joint probability that a respondent is from France, earns less than 100,000 USD annually, and uses Cross-Validation Often or Most of the time:", france_lt_100k_cross_prob)


Joint probability that a respondent is from France, earns less than 100,000 USD annually, and uses Cross-Validation Often or Most of the time: 0.020478723404255317


In [145]:
#from 2a) pr_given_cpp, pr_and_cpp, cpp_users

pr = len(mcr[mcr['CurrentJobTitleSelect'] == "Programmer"])

cpp_given_pr = (pr_given_cpp * cpp_users) / pr

print(cpp_given_pr)

0.0735930735930736
